<a href="https://colab.research.google.com/github/brunooseliero/tcc_uspesalq/blob/main/script_tcc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# ETAPA 1: ENGENHARIA DE DADOS (ETL) E CONSTRUÇÃO DO DATASET
# AVISO AO AVALIADOR: Esta etapa realiza downloads extensivos da API do Yahoo
# Finance. A execução completa deste bloco pode levar de 5 a 10 minutos.
# ==============================================================================

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# 1. Definição da amostra: 63 Tickers do Ibovespa
tickers_ibov = [
    'ALOS3', 'ABEV3', 'ASAI3', 'AZUL4', 'B3SA3', 'BBSE3', 'BBDC4', 'BBAS3',
    'BEEF3', 'BPAC11', 'BRAP4', 'BRKM5', 'BRAV3', 'CMIG4', 'CPLE5', 'CSAN3',
    'CPFE3', 'CMIN3', 'CYRE3', 'DXCO3', 'ELET3', 'ELET6', 'EMBR3', 'ENGI11',
    'ENEV3', 'EGIE3', 'EQTL3', 'EZTC3', 'FLRY3', 'GGBR4', 'GOAU4', 'HAPV3',
    'HYPE3', 'IGTI11', 'ITSA4', 'ITUB4', 'KLBN11', 'LREN3', 'MGLU3', 'MRVE3',
    'MULT3', 'PCAR3', 'PETR4', 'PRIO3', 'PETZ3', 'RADL3', 'RAIL3', 'RDOR3',
    'RENT3', 'SANB11', 'CSNA3', 'SBSP3', 'SLCE3', 'SMTO3', 'SUZB3', 'TAEE11',
    'TIMS3', 'TOTS3', 'UGPA3', 'USIM5', 'VALE3', 'VIVT3', 'WEGE3'
]
tickers_fmt = [f"{t}.SA" for t in tickers_ibov]

# 2. Coleta de Cotações Históricas e Cálculo de Volatilidade e Retorno
print("Iniciando download de cotações e cálculo do Target...")
dados_list = []
for t in tickers_fmt:
    try:
        # Download de 2014 a 2024
        df_temp = yf.download(t, start="2014-01-01", end="2024-12-31", progress=False)
        if not df_temp.empty:
            if isinstance(df_temp.columns, pd.MultiIndex):
                df_temp.columns = df_temp.columns.get_level_values(0)

            df_temp['Retorno_Diario'] = df_temp['Close'].pct_change()
            df_temp['Ano'] = df_temp.index.year

            # Anualização da volatilidade e retorno composto
            anual = df_temp.groupby('Ano').agg({
                'Retorno_Diario': ['std', lambda x: (1+x).prod() - 1]
            })
            anual.columns = ['Volatilidade_Anual', 'Retorno_Anual']
            anual['Volatilidade_Anual'] = anual['Volatilidade_Anual'] * np.sqrt(252)
            anual['Ticker'] = t
            dados_list.append(anual.reset_index())
    except: pass

df_precos = pd.concat(dados_list)

# Coleta do Benchmark (Ibovespa) para definição do Target Binário
try:
    ibov = yf.download("^BVSP", start="2014-01-01", end="2024-12-31", progress=False)
    if isinstance(ibov.columns, pd.MultiIndex): ibov.columns = ibov.columns.get_level_values(0)
    ibov['Retorno'] = ibov['Close'].pct_change()
    ibov_anual = ibov.groupby(ibov.index.year)['Retorno'].apply(lambda x: (1+x).prod() - 1).reset_index()
    ibov_anual.columns = ['Ano', 'Retorno_Ibov']

    df_precos = pd.merge(df_precos, ibov_anual, on='Ano', how='left')
    df_precos['Target_Venceu'] = (df_precos['Retorno_Anual'] > df_precos['Retorno_Ibov']).astype(int)
except:
    print("Aviso: Erro ao baixar Ibovespa. Verifique a conexão.")

# 3. Coleta de Fundamentos e Criação de Features
print("Iniciando download de demonstrativos contábeis...")
fund_list = []
for t in tickers_fmt:
    try:
        stock = yf.Ticker(t)
        dre = stock.income_stmt.T
        bp = stock.balance_sheet.T
        fc = stock.cashflow.T

        if not dre.empty and not bp.empty:
            full = dre.join(bp, how='outer', lsuffix='_D', rsuffix='_B').join(fc, how='outer', rsuffix='_F')
            full['Ano'] = full.index.year
            full['Ticker'] = t

            # Função de extração segura
            def get_col(df, candidates):
                for c in candidates:
                    if c in df.columns: return pd.to_numeric(df[c], errors='coerce')
                return pd.Series(0, index=df.index)

            # Extração de contas contábeis
            full['EBIT'] = get_col(full, ['EBIT', 'Operating Income'])
            full['Patrimonio'] = get_col(full, ['Stockholders Equity', 'Total Equity Gross Minority Interest'])
            full['Divida'] = get_col(full, ['Total Debt', 'Total Financial Debt'])
            full['Caixa'] = get_col(full, ['Cash And Cash Equivalents'])
            full['Lucro'] = get_col(full, ['Net Income', 'Net Income Common Stockholders'])
            full['CAPEX'] = get_col(full, ['Capital Expenditure'])
            full['Receita'] = get_col(full, ['Total Revenue', 'Operating Revenue'])
            full['Lucro_Bruto'] = get_col(full, ['Gross Profit'])
            full['Ativo_Total'] = get_col(full, ['Total Assets'])
            full['Caixa_Operacional'] = get_col(full, ['Operating Cash Flow', 'Cash Flow From Continuing Operating Activities'])

            # Ordenação temporal necessária para o Piotroski (Deltas)
            full = full.sort_index(ascending=True)

            # ---  MÉTRICAS BÁSICAS ---
            full['ROIC'] = (full['EBIT'] * (1 - 0.34)) / (full['Divida'] + full['Patrimonio'])
            full['DivLiq_EBITDA'] = (full['Divida'] - full['Caixa']) / full['EBIT']
            full['ROE'] = full['Lucro'] / full['Patrimonio']

            # --- PIOTROSKI F-SCORE MODIFICADO (7 Critérios) ---
            full['ROA'] = full['Lucro'] / full['Ativo_Total']
            full['Alavancagem_Ativo'] = full['Divida'] / full['Ativo_Total']
            full['Margem_Bruta'] = full['Lucro_Bruto'] / full['Receita']
            full['Giro_Ativo'] = full['Receita'] / full['Ativo_Total']
            full['FCF'] = full['Caixa_Operacional'] + full['CAPEX'] # CAPEX geralmente é negativo no Yahoo

            # Variações ano contra ano
            full['ROA_Anterior'] = full['ROA'].shift(1)
            full['Alavancagem_Anterior'] = full['Alavancagem_Ativo'].shift(1)
            full['Margem_Anterior'] = full['Margem_Bruta'].shift(1)
            full['Giro_Anterior'] = full['Giro_Ativo'].shift(1)

            full['F_Score'] = (
                (full['ROA'] > 0).astype(int) +
                (full['FCF'] > 0).astype(int) +
                (full['FCF'] > full['Lucro']).astype(int) +
                (full['ROA'] > full['ROA_Anterior']).astype(int) +
                (full['Alavancagem_Ativo'] < full['Alavancagem_Anterior']).astype(int) +
                (full['Margem_Bruta'] > full['Margem_Anterior']).astype(int) +
                (full['Giro_Ativo'] > full['Giro_Anterior']).astype(int)
            )

            fund_list.append(full[['Ticker', 'Ano', 'ROIC', 'F_Score', 'DivLiq_EBITDA', 'ROE']])
    except Exception as e:
        pass

df_fund = pd.concat(fund_list)

# 4. Dados Macroeconômicos e Governança Corporativa
macro_data = {
    'Ano': [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'Selic_Media': [0.109, 0.142, 0.142, 0.100, 0.065, 0.059, 0.027, 0.044, 0.123, 0.130, 0.105],
    'IPCA_Anual': [0.064, 0.106, 0.062, 0.029, 0.037, 0.043, 0.045, 0.100, 0.057, 0.046, 0.045]
}
df_macro = pd.DataFrame(macro_data)

# Consolidação
df_final = pd.merge(df_precos, df_fund, on=['Ticker', 'Ano'], how='inner')
df_final = pd.merge(df_final, df_macro, on='Ano', how='left')

# --- INTERAÇÃO MACRO-MICRO ---
# Em vez de manter Selic estática, criamos variáveis de interação
df_final['ROE_Ajustado_Selic'] = df_final['ROE'] / df_final['Selic_Media']
df_final['Custo_Divida_Real'] = df_final['DivLiq_EBITDA'] * df_final['Selic_Media']

# Remoção das variáveis macro isoladas
df_final.drop(columns=['Selic_Media', 'IPCA_Anual'], inplace=True, errors='ignore')

# Classificação qualitativa
estatais = ['PETR4.SA', 'PETR3.SA', 'BBAS3.SA', 'CMIG4.SA', 'ELET3.SA', 'ELET6.SA']
df_final['Eh_Estatal'] = df_final['Ticker'].apply(lambda x: 1 if x in estatais else 0)
novo_mercado = ['WEGE3.SA', 'LREN3.SA', 'MGLU3.SA', 'VALE3.SA', 'B3SA3.SA', 'RENT3.SA', 'RADL3.SA']
df_final['Eh_Novo_Mercado'] = df_final['Ticker'].apply(lambda x: 1 if x in novo_mercado else 0)

# 5. Salvamento do Dataset Final
df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
df_final.dropna(inplace=True)
df_final.to_csv('DATABASE_MESTRE_FINAL.csv', index=False)
print(f"Base de dados gerada com sucesso! Total de linhas válidas: {len(df_final)}")

In [ ]:
# ==============================================================================
# ETAPA 2: MACHINE LEARNING, DEEP LEARNING E TESTES ESTATÍSTICOS (CÓDIGO COMPLETO)
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.utils import resample
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

# 1. Carregamento do dataset gerado na Etapa 1
try:
    df = pd.read_csv('DATABASE_MESTRE_FINAL.csv')
    print("Dataset carregado com sucesso!")
except FileNotFoundError:
    print("Erro: Arquivo DATABASE_MESTRE_FINAL.csv não encontrado.")

# --- CORREÇÃO RÁPIDA: Reinjetando a Selic ---
selic_dict = {
    2014: 0.109, 2015: 0.142, 2016: 0.142, 2017: 0.100, 2018: 0.065,
    2019: 0.059, 2020: 0.027, 2021: 0.044, 2022: 0.123, 2023: 0.130, 2024: 0.105
}
df['Selic_Media'] = df['Ano'].map(selic_dict)

# --- RECUPERAÇÃO DE ROE E ROIC CASO TENHAM SIDO EXCLUÍDOS DO CSV ---
if 'ROE' not in df.columns:
    try:
        df['ROE'] = df['Lucro'] / df['Patrimonio']
    except KeyError:
        print("Aviso: Colunas base para ROE não encontradas.")

if 'ROIC' not in df.columns:
    try:
        df['ROIC'] = (df['EBIT'] * (1 - 0.34)) / (df['Divida'] + df['Patrimonio'])
    except KeyError:
        print("Aviso: Colunas base para ROIC não encontradas.")

# --- CORREÇÃO 1: VARIÁVEIS MACRO REAIS (EXCESS RETURN) ---
# Criamos os prêmios de risco
if 'ROE' in df.columns:
    df['ROE_Premio_Risco'] = df['ROE'] - df['Selic_Media']
if 'ROIC' in df.columns:
    df['ROIC_Premio_Risco'] = df['ROIC'] - df['Selic_Media']

# --- AQUI ESTÁ O TRUQUE PARA EVITAR A MULTICOLINEARIDADE ---
# NÃO colocamos ROE nem ROIC puros na lista de features, APENAS os prêmios de risco
features_esperadas = [
    'Volatilidade_Anual', 'F_Score', 'DivLiq_EBITDA',
    'ROE_Premio_Risco', 'ROIC_Premio_Risco', 'Eh_Estatal', 'Eh_Novo_Mercado'
]

# Filtramos apenas as colunas que realmente existem para evitar erros
features = [f for f in features_esperadas if f in df.columns]

X = df[features].copy()
y = df['Target_Venceu'].copy()

# Tratamento de nulos/infinitos
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())

# 2. Divisão Cronológica Treino/Teste
X['Ano'] = df['Ano']
X = X.sort_values(by='Ano')
y = y.loc[X.index]

ano_corte = 2021
train_idx = X['Ano'] <= ano_corte
test_idx = X['Ano'] > ano_corte

X = X.drop(columns=['Ano'])

X_train = X[train_idx]
y_train = y[train_idx]
X_test = X[test_idx]
y_test = y[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Modelos Tradicionais
modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
}

# --- CORREÇÃO 2: GRID SEARCH COM TIME SERIES SPLIT ---
print("\nOtimizando MLP com TimeSeriesSplit para evitar Data Leakage...")
tscv = TimeSeriesSplit(n_splits=3)
mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100, 50)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.01]
}
mlp_base = MLPClassifier(max_iter=1000, random_state=42, early_stopping=True)
grid_mlp = GridSearchCV(mlp_base, mlp_param_grid, cv=tscv, scoring='roc_auc', n_jobs=-1)
grid_mlp.fit(X_train_scaled, y_train)

modelos["Deep Learning (MLP)"] = grid_mlp.best_estimator_

# 4. Avaliação e Métricas
resultados = []
probs_dict = {}

print("\n=== RELATÓRIO DE PERFORMANCE DETALHADO (PRECISION/RECALL) ===")
for nome, modelo in modelos.items():
    if nome != "Deep Learning (MLP)":
        modelo.fit(X_train_scaled, y_train)

    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    y_pred = modelo.predict(X_test_scaled)
    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)

    probs_dict[nome] = y_prob

    print(f"\n--- {nome} ---")
    print(f"ROC-AUC: {auc:.4f} | Acurácia: {acc:.4f}")
    print(classification_report(y_test, y_pred))

    resultados.append({"Modelo": nome, "ROC-AUC": auc, "Acurácia": acc})

# --- CORREÇÃO 3: TESTE DE SIGNIFICÂNCIA ESTATÍSTICA (BOOTSTRAP) ---
print("\n=== TESTE DE SIGNIFICÂNCIA (BOOTSTRAP 95% CI PARA ROC-AUC) ===")
n_bootstraps = 1000
for nome in ["Random Forest", "XGBoost"]:
    bootstrapped_scores = []
    y_test_np = y_test.values
    prob_np = probs_dict[nome]

    for i in range(n_bootstraps):
        indices = resample(np.arange(len(y_test_np)), random_state=i)
        if len(np.unique(y_test_np[indices])) < 2:
            continue
        score = roc_auc_score(y_test_np[indices], prob_np[indices])
        bootstrapped_scores.append(score)

    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    lower = sorted_scores[int(0.025 * len(sorted_scores))]
    upper = sorted_scores[int(0.975 * len(sorted_scores))]
    print(f"{nome}: Intervalo de Confiança 95% = [{lower:.4f} - {upper:.4f}]")

# 5. Tabelas Finais e Gráficos
df_resultados = pd.DataFrame(resultados).sort_values(by="ROC-AUC", ascending=False)

print("\n=== NOVA IMPORTÂNCIA DAS VARIÁVEIS ===")
rf_treinado = modelos["Random Forest"]
df_importancia = pd.DataFrame({
    'Variável': features,
    'Importância': rf_treinado.feature_importances_
}).sort_values(by='Importância', ascending=False)
print(df_importancia)

print("\n=== NOVA TABELA DE ESTATÍSTICA DESCRITIVA ===")
# Removemos as colunas binárias para a tabela descritiva ficar limpa
cols_desc = [c for c in X_train.columns if c not in ['Eh_Estatal', 'Eh_Novo_Mercado']]
estatisticas = X_train[cols_desc].describe().T[['mean', 'std', 'min', '50%', 'max']]
estatisticas.columns = ['Média', 'Desvio Padrão', 'Mínimo', 'Mediana', 'Máximo']
print(estatisticas.round(3))

# --- 6. GERAÇÃO E SALVAMENTO DE TODOS OS GRÁFICOS ---
sns.set_theme(style="whitegrid")

# Gráfico 1: Heatmap (Correlação limpa, sem Multicolinearidade)
plt.figure(figsize=(10, 8))
corr_matrix = X_train[cols_desc].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Matriz de Correlação das Variáveis Preditivas', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('grafico_correlacao.png', dpi=300)

# Gráfico 2: Desempenho dos Modelos
plt.figure(figsize=(10, 5))
sns.barplot(x="ROC-AUC", y="Modelo", data=df_resultados, palette="Blues_r")
plt.title('Desempenho dos Modelos Preditivos (ROC-AUC)', fontsize=14, pad=15)
plt.xlim(0, 1)
plt.xlabel('Área Sob a Curva ROC (AUC)')
plt.ylabel('')
plt.tight_layout()
plt.savefig('grafico_desempenho_modelos.png', dpi=300)

# Gráfico 3: Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Importância', y='Variável', data=df_importancia, palette="magma")
plt.title('Importância das Variáveis (Random Forest)', fontsize=14, pad=15)
plt.xlabel('Nível de Importância Relativa')
plt.ylabel('Variáveis Preditivas')
plt.tight_layout()
plt.savefig('grafico_feature_importance.png', dpi=300)

print("\nProcesso finalizado com sucesso! Imagens geradas e salvas na sua pasta.")